# Import das bibliotecas

In [1]:
!pip install polars --quiet

In [2]:
import pandas as pd 
import math 
import warnings 
import numpy as np
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# Geração e enriquecimento da base

In [3]:
def gerar_base_agro(N=50_000, start='2024-01-01', culturas=('soja', 'milho', 'cafe'), regioes=('Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul'), seed=42): 
    '''
    Gerar dados sintéticos com variáveis: 
    N, P, K, temperatua, umidade, ph, chuva, label(binário), cultura e região. 
    A label é determinado por quão perto os atributos estão do 'ótimo' de cada cultura
    '''

    rng = np.random.default_rng(seed)
    data_inicio = pd.to_datetime(start)
    timestamps = pd.date_range(data_inicio, periods=N, freq='10min')

    #culturas sorteadas com pesos 
    culturas = np.array(culturas)
    regioes = np.array(regioes)
    cult = rng.choice(culturas, size=N, p=[0.4, 0.4, 0.2])
    reg = rng.choice(regioes, size=N, p=[0.2, 0.25, 0.15, 0.25, 0.15])
    
    # define estação a partir do mês
    meses = pd.to_datetime(timestamps).month 
    estacao = np.select(
        [meses.isin([12,1,2]), meses.isin([3,4,5]), meses.isin([6,7,8]), meses.isin([9,10,11])],
        ['verao','outono','inverno','primavera'], 
        default='desconhecida'
    ).astype(str)

    # condições médias por região (°C, umidade %, chuva base mm, pH shift)
    condicoes_regiao = {
        'Norte':        {'temp': 28, 'umid': 75, 'chuva': 4.5, 'ph_shift': -0.4},
        'Nordeste':     {'temp': 30, 'umid': 65, 'chuva': 2.5, 'ph_shift': -0.3},
        'Centro-Oeste': {'temp': 27, 'umid': 55, 'chuva': 2.0, 'ph_shift': -0.1},
        'Sudeste':      {'temp': 24, 'umid': 60, 'chuva': 3.0, 'ph_shift': 0.0},
        'Sul':          {'temp': 21, 'umid': 68, 'chuva': 3.2, 'ph_shift': +0.1},
    }

    # estação afeta chuva (fator multiplicativo)
    fator_chuva_estacao = {'verao':1.3,'outono':0.9,'inverno':0.6,'primavera':1.1}

    # ótimos aproximados de cada cultura
    otimos = {
        'soja': {'N':70,'P':50,'K':60,'temp':26,'umid':38,'pH_min':6.0,'pH_max':6.8,'chuva_max':1.5},
        'milho':{'N':80,'P':45,'K':65,'temp':25,'umid':40,'pH_min':5.8,'pH_max':7.0,'chuva_max':2.0},
        'cafe': {'N':60,'P':40,'K':55,'temp':22,'umid':42,'pH_min':5.5,'pH_max':6.5,'chuva_max':2.5},
    }

     # --- geração base ---
    Nn = np.clip(rng.normal(65, 18, N), 0, 100)
    Pp = np.clip(rng.normal(48, 15, N), 0, 100)
    Kk = np.clip(rng.normal(60, 15, N), 0, 100)

    # temperatura e umidade regionais
    temp = np.zeros(N)
    umid = np.zeros(N)
    chuva = np.zeros(N)
    pH = np.zeros(N)

    for r in condicoes_regiao: 
        idx = (reg == r) 
        base = condicoes_regiao[r]
        f_chuva = np.array([fator_chuva_estacao[e] for e in estacao[idx]])
        temp[idx] = np.clip(rng.normal(base['temp'], 3, idx.sum()), 5, 45)
        umid[idx] = np.clip(rng.normal(base['umid'], 10, idx.sum()), 10, 100)
        chuva[idx] = np.clip(rng.exponential(base['chuva'], idx.sum()) * f_chuva, 0, 12)
        pH[idx] = np.clip(rng.normal(6.4 + base['ph_shift'], 0.6, idx.sum()), 4.5, 8.5)

    # Label - condição ideal 
    prob = np.zeros(N)
    for c in np.unique(cult): 
        idx = (cult == c) 
        ot = otimos[c]
        dN = np.abs(Nn[idx] - ot['N']) / 25
        dP = np.abs(Pp[idx] - ot['P']) / 20
        dK = np.abs(Kk[idx] - ot['K']) / 20
        dT = np.abs(temp[idx] - ot['temp']) / 8
        dU = np.abs(umid[idx] - ot['umid']) / 12
        dPH = np.where((pH[idx] >= ot['pH_min']) & (pH[idx] <= ot['pH_max']), 0, 1)
        dR = np.clip(chuva[idx] / ot['chuva_max'], 0, 2)
        score = 3.5 - (dN + dP + dK + dT + dU) - 0.8*dPH - 0.5*(dR>1.2)
        prob[idx] = 1 / (1 + np.exp(-score))

    # ruído + ajuste climático leve 
    prob = np.clip(prob + rng.normal(0, 0.05, N), 0, 1)
    label = (rng.random(N) < prob).astype(int)

    df = pd.DataFrame({
        'timestamp': timestamps,
        'cultura': cult,
        'regiao': reg,
        'estacao': estacao,
        'N': Nn.round(1),
        'P': Pp.round(1),
        'K': Kk.round(1),
        'temperatura': temp.round(1),
        'umidade': umid.round(1),
        'pH': pH.round(2),
        'chuva': chuva.round(2),
        'label': label
    })

    return df 

In [4]:
def add_produtividade(df: pd.DataFrame, seed: int=42) -> pd.DataFrame: 
    '''
    Adiciona a coluna 'produtividade_kg_ha' com base em: 
    - cultura (rendimento base) 
    - região e estação (ajustes climáticos) 
    - adequação nutricional e climática (score)
    - ruído aleatório para realismo 
    '''

    rng = np.random.default_rng(seed)
    df = df.copy()

    # Rendimento base por cultura (kg/ha)
    base_yield = { 
        'soja': 3800, 
        'milho': 7500, 
        'cafe': 1800
    }
    base = df['cultura'].map(base_yield).fillna(4000)

    # Ajustes por região (percentual)
    ajuste_reg = { 
        'Norte': -0.05, 'Nordeste':-0.10, 
        'Centro-Oeste':0.00, 'Sudeste': +0.05, 'Sul': +0.08
    }
    adj_r = df['regiao'].map(ajuste_reg).fillna(0)

    # Ajuste por estação (percentual)
    ajuste_est = {'verao': +0.07, 'outono': +0.00, 'inverno': -0.08, 'primavera': +0.03}
    adj_e = df['estacao'].map(ajuste_est).fillna(0)

    # Fator de adequação (0-1) derivado das variáveis ambientais 
    # Pesos ajustados empiricamente 
    n_opt, p_opt, k_opt = 70, 50, 60
    ph_centro, ph_tol = 6.4, 1.0
    temp_centro, temp_tol = 26, 10
    umid_centro, umid_tol = 40, 15
    chuva_max = 3.0

     # Distâncias normalizadas
    dN = np.abs(df['N'] - n_opt) / 25
    dP = np.abs(df['P'] - p_opt) / 20
    dK = np.abs(df['K'] - k_opt) / 20
    dPH = np.abs(df['pH'] - ph_centro) / ph_tol
    dT = np.abs(df['temperatura'] - temp_centro) / temp_tol
    dU = np.abs(df['umidade'] - umid_centro) / umid_tol
    dR = np.clip(df['chuva'] / chuva_max, 0, 2)

    # Score combinado (menor distância = melhor rendimento)
    score = 3.5 - (dN + dP + dK + 0.6*dPH + 0.6*dT + 0.5*dU) - 0.5*(dR > 1.2)
    adequacao = 1 / (1 + np.exp(-score))  # sigmoide [0,1]
    adequacao = np.clip(adequacao, 0, 1)

    # --- Produtividade final (com ruído e ajustes) ---
    ruido = rng.normal(1.0, 0.08, len(df))  # ruído 8% gaussiano
    fator_final = (1 + adj_r + adj_e) * adequacao * ruido
    df['produtividade_kg_ha'] = np.round(base * fator_final, 2)

    # --- Limites de plausibilidade ---
    # (mínimo 50% da base, máximo 140%)
    df['produtividade_kg_ha'] = np.clip(df['produtividade_kg_ha'], base * 0.5, base * 1.4)

    return df


### Gerar a base a principio para visualização e adicionar a coluna de produtividade

In [5]:
df = gerar_base_agro()

In [6]:
df.tail()

,timestamp,cultura,regiao,estacao,N,P,K,temperatura,umidade,pH,chuva,label
49995,2024-12-13 04:30:00,cafe,Nordeste,verao,67.2,57.5,44.2,24.3,85.8,5.71,3.30,0
49996,2024-12-13 04:40:00,milho,Centro-Oeste,verao,56.5,55.6,70.0,30.3,64.6,5.39,10.20,0
49997,2024-12-13 04:50:00,cafe,Centro-Oeste,verao,70.4,49.1,61.7,24.6,51.3,6.30,6.59,1
49998,2024-12-13 05:00:00,milho,Norte,verao,57.2,16.5,48.4,27.0,84.6,5.93,12.00,0
49999,2024-12-13 05:10:00,cafe,Sul,verao,50.0,23.8,71.3,22.4,71.0,6.45,1.85,0


In [7]:
df = add_produtividade(df)

In [8]:
df.tail()

,timestamp,cultura,regiao,estacao,N,P,K,temperatura,umidade,pH,chuva,label,produtividade_kg_ha
49995,2024-12-13 04:30:00,cafe,Nordeste,verao,67.2,57.5,44.2,24.3,85.8,5.71,3.30,0,973.14
49996,2024-12-13 04:40:00,milho,Centro-Oeste,verao,56.5,55.6,70.0,30.3,64.6,5.39,10.20,0,4588.63
49997,2024-12-13 04:50:00,cafe,Centro-Oeste,verao,70.4,49.1,61.7,24.6,51.3,6.30,6.59,1,1811.71
49998,2024-12-13 05:00:00,milho,Norte,verao,57.2,16.5,48.4,27.0,84.6,5.93,12.00,0,3750.00
49999,2024-12-13 05:10:00,cafe,Sul,verao,50.0,23.8,71.3,22.4,71.0,6.45,1.85,0,900.00


# Salvar em parquet se for possível devido a utilização do python 3.14 

- Anteriormente houve quebra de incompatibilidade em projetos ao salvar no arquivo parquet e particionado para uso posteriormente no treinamento tirando todo o peso da memória

In [9]:
import os
def salvar_particionamento(df: pd.DataFrame, root_dir: str = 'data', max_linhas_por_parte=1_000_000): 
    '''
    Tenta salvar em parquet, se não conseguir, faz fallback para csv
    '''
    df = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']): 
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['ano'] = df['timestamp'].dt.year
    df['mes'] = df['timestamp'].dt.month

    has_polars = False 
    try: 
        import polars as pl 
        has_polars = True 
    except Exception: 
        pass

    has_pyarrow = False
    try:
        import pyarrow
        has_pyarrow = True
    except Exception:
        pass

    has_fastparquet = False 
    try: 
        import fastparquet
        has_fastparquet = True 
    except Exception: 
        pass 

    def _save_parquet_pandas(g: pd.DataFrame, out_path: str, engine: str): 
        g.drop(columns=['ano', 'mes']).to_parquet(out_path, index=False, engine=engine)

    for (ano, mes), g in df.groupby(['ano', 'mes'], sort=True, as_index=False):
        n = len(g)
        n_parts = math.ceil(n / max_linhas_por_parte)
        out_dir = os.path.join(root_dir, f'ano={ano}', f'mes={mes:02d}')
        os.makedirs(out_dir, exist_ok=True)
        
        for i, idx in enumerate(range(0, n, max_linhas_por_parte), start=1): 
            chunk = g.iloc[idx: idx+max_linhas_por_parte].copy()

            if has_polars: 
                try: 
                    import polars as pl 
                    out_path = os.path.join(out_dir, f'part-{i:04d}.parquet')
                    pl.from_pandas(chunk.drop(columns=['ano', 'mes'])).write_parquet(out_path)
                    continue
                except Exception: 
                    pass

            if has_pyarrow: 
                try:
                    out_path = os.path.join(out_dir, f'part-{i:04d}.parquet')
                    _save_parquet_pandas(chunk, out_path, engine='pyarrow')
                    continue
                except Exception:
                    pass

            if has_fastparquet:
                try:
                    out_path = os.path.join(out_dir, f'part-{i:04d}.parquet')
                    _save_parquet_pandas(chunk, out_path, engine='fastparquet')
                    continue
                except Exception:
                    pass
            
            out_path = os.path.join(out_dir, f'part-{i:04d}.csv.gz')
            chunk.drop(columns=['ano','mes']).to_csv(out_path, index=False, compression='gzip')

    print('OK: dados gravados em', root_dir)
    print('Formato preferido: Parquet (se backend disponível); caso contrário, CSV .gz.')

In [10]:
salvar_particionamento(df, root_dir='data')

OK: dados gravados em data
Formato preferido: Parquet (se backend disponível); caso contrário, CSV .gz.


In [11]:
import polars as pl 
lf = pl.scan_parquet('data/ano=*/mes=*/part-*.parquet')
df_pl = lf.collect()
df_pl.head()

timestamp,cultura,regiao,estacao,N,P,K,temperatura,umidade,pH,chuva,label,produtividade_kg_ha
datetime[ns],str,str,str,f64,f64,f64,f64,f64,f64,f64,i64,f64
2024-01-01 00:00:00,"""milho""","""Sul""","""verao""",84.2,66.5,41.6,18.6,69.9,6.2,12.0,0,3750.0
2024-01-01 00:10:00,"""milho""","""Nordeste""","""verao""",41.5,39.8,58.8,33.5,49.4,5.77,12.0,0,3750.0
2024-01-01 00:20:00,"""cafe""","""Sudeste""","""verao""",75.0,46.6,37.4,22.9,53.7,6.4,2.82,1,1699.64
2024-01-01 00:30:00,"""milho""","""Norte""","""verao""",65.2,58.0,65.5,26.8,84.5,6.22,12.0,0,5107.87
2024-01-01 00:40:00,"""soja""","""Centro-Oeste""","""verao""",100.0,32.8,54.0,29.0,64.5,5.21,5.79,0,1900.0


# EDA da Produtividade
- Foco: Validar a variável produtividade_kg_ha, checar coerência e relações com NPK, pH, clima, região e estação

In [12]:
import matplotlib.pyplot as plt 

def eda_produtividade(df: pd.DataFrame, out_dir:str = 'eda_produtividade', amostra:int = 500_000): 
    '''
    Gera 6 gráficos em PNG para explorar 'produtividade_kg_ha'
    Usando 100% matplotlib 
    '''
    os.makedirs(out_dir, exist_ok=True)
    if len(df) > amostra: 
        d = df.sample(amostra, random_state=1).copy()
    else: 
        d = df.copy()

    # 1. Histograma de produtividade (geral)
    plt.figure(figsize=(8, 4.5))
    plt.hist(d['produtividade_kg_ha'], bins=60)
    plt.title('Distribuição da Produtividade (kg/ha)')
    plt.xlabel('Produtividade (kg/ha)')
    plt.ylabel('Frequência')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, '01_hist_produtividade.png'))
    plt.close()

    # 2. Histograma por cultura (arquivos separados)
    for c in d['cultura'].unique(): 
        plt.figure(figsize=(8, 4.5))
        plt.hist(d.loc[d['cultura'] == c, 'produtividade_kg_ha'], bins=60)
        plt.title(f'Produtividade por Cultura: {c}')
        plt.xlabel('Produtividade (kg/ha)')
        plt.ylabel('Frequência')
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'02_hist_prod_{c}.png'))
        plt.close()

    # 3. Boxplot da produtividade por região 
    # preparar dados como listas por categoria 
    data_reg = [d.loc[d['regiao']==r, 'produtividade_kg_ha'].values for r in sorted(d['regiao'].unique())]
    plt.figure(figsize=(9, 4.8))
    plt.boxplot(data_reg, showfliers=False)
    plt.xticks(range(1, len(data_reg)+1), sorted(d['regiao'].unique()), rotation=0)
    plt.title('Produtividade por Região (sem outliers)')
    plt.ylabel('Produtividade (kg/ha)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, '03_box_regiao.png'))
    plt.close()

    # 4. Boxplot da produtividade por estação
    ordem_est = ['verao', 'outono', 'inverno', 'primavera']
    data_est = [d.loc[d['estacao']==e, 'produtividade_kg_ha'].values for e in ordem_est if e in set(d['estacao'].unique())]
    est_labels = [e for e in ordem_est if e in set(d['estacao'].unique())]
    plt.figure(figsize=(9, 4.8))
    plt.boxplot(data_est, showfliers=False)
    plt.xticks(range(1, len(est_labels)+1), est_labels, rotation=0)
    plt.title('Produtividade por Estação (sem outliers)')
    plt.ylabel('Produtividade (kg/ha)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, '04_box_estacao.png'))
    plt.close()

    # 5. Dispersões produtividade vs N e vs pH
    # 5.1 Produtividade vs N 
    plt.figure(figsize=(8,4.5))
    plt.scatter(d['N'], d['produtividade_kg_ha'], s=4, alpha=0.25)
    plt.title('Produtividade vs N')
    plt.xlabel('N')
    plt.ylabel('Produtividade (kg/ha)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, '05_scatter_prod_vs_N.png'))
    plt.close()

    # 5.2 Produtividade vs pH
    plt.figure(figsize=(8,4.5))
    plt.scatter(d['pH'], d['produtividade_kg_ha'], s=4, alpha=0.25)
    plt.title('Produtividade vs pH')
    plt.xlabel('pH')
    plt.ylabel('Produtividade (kg/ha)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, '06_scatter_prod_vs_pH.png'))
    plt.close()

    # 6. Heatmap de correlação (inclui produtividade)
    num_cols = ['N','P','K','temperatura','umidade','pH','chuva','label','produtividade_kg_ha']
    d_num = d[num_cols].copy()
    corr = d_num.corr(numeric_only=True)

    plt.figure(figsize=(7.5,6))
    im = plt.imshow(corr, interpolation='nearest')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha='right')
    plt.yticks(range(len(num_cols)), num_cols)
    plt.title('Matriz de Correlação (incl. produtividade)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, '07_corr_heatmap.png'))
    plt.close()

    print(f'OK: EDA salva em "{out_dir}"')


In [13]:
eda_produtividade(df, out_dir='eda_produtividade', amostra=500_000)


OK: EDA salva em "eda_produtividade"


# Feature Engineering + split temporal

In [14]:
!pip install scikit-learn --quiet

In [15]:
def build_features_reg(df: pd.DataFrame): 
    d = df
    if not pd.api.types.is_datetime64_any_dtype(d['timestamp']): 
        d['timestamp'] = pd.to_datetime(d['timestamp'])
    
    d['mes'] = d['timestamp'].dt.month
    d['mes_sin'] = np.sin(2*np.pi*d['mes']/12)
    d['mes_cos'] = np.cos(2*np.pi*d['mes']/12)

    # one-hot leve para categorias
    d = pd.get_dummies(d, columns=['cultura', 'regiao', 'estacao'], drop_first=True)

    # Ordernar por tempo e faz split temporal (treino anos anteriores, teste posteriores)
    d = d.sort_values('timestamp').reset_index(drop=True)
    corte = int(len(d) * 0.8)
    d_train, d_test = d.iloc[:corte], d.iloc[corte:]

    feats_base = ['N','P','K','temperatura','umidade','pH','chuva','mes_sin','mes_cos']
    feats_cat = [c for c in d.columns if c.startswith('cultura_') or c.startswith('regiao_') or c.startswith('estacao_')]
    feats = feats_base + feats_cat

    X_train = d_train[feats].values
    y_train = d_train['produtividade_kg_ha'].values
    X_test = d_test[feats].values 
    y_test = d_test['produtividade_kg_ha'].values

    meta = {
        "feats": feats,
        "n_train": len(d_train),
        "n_test": len(d_test),
        "corte_timestamp": (d_train["timestamp"].iloc[-1], d_test["timestamp"].iloc[0]),
        "cols_cat": feats_cat
    }

    return (X_train, y_train, X_test, y_test), meta, d_train, d_test

# Treino dos 5 modelos, métricas e ranking

In [16]:
import time 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [20]:
try:
    from sklearn.metrics import root_mean_squared_error
    _HAS_RMSE = True
except Exception:
    _HAS_RMSE = False

def avaliar(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = root_mean_squared_error(y_true, y_pred) if _HAS_RMSE else np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return rmse, mae, r2

def treinar_modelos_reg(X_train, y_train, X_test, y_test, n_jobs=1): 
    modelos = { 
        'LinearRegression': Pipeline([('scaler', StandardScaler(with_mean=True, with_std=True)), 
                                      ('mdl', LinearRegression(n_jobs=None))]),

        'RandomForestRegressor': RandomForestRegressor(n_estimators=120, max_depth=None, n_jobs=n_jobs, random_state=42), 

        'GradientBoostingRegressor': Pipeline([('scaler', GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=3, random_state=42))]), 

        'KNeighborsRegressor': Pipeline([("scaler", StandardScaler()),
                                         ("mdl", KNeighborsRegressor(n_neighbors=25))]),

        'SVR-RBF': Pipeline([('scaler', StandardScaler()), 
                             ('mdl', SVR(kernel='rbf', C=3.0, gamma='scale'))])
    }

    resultados = []
    modelos_fit = {}

    for nome, mdl in modelos.items(): 
        t0 = time.time()
        mdl.fit(X_train, y_train)
        y_pred = mdl.predict(X_test)
        rmse, mae, r2 = avaliar(y_test, y_pred)
        dt = time.time() - t0

        resultados.append({'modelo': nome, 'RMSE':rmse, 'MAE':mae, 'R2':r2, 'tempo_s':dt})
        modelos_fit[nome] = mdl

    resultados = pd.DataFrame(resultados).sort_values(by=['RMSE', 'MAE', 'R2'], ascending=[True, True, False]).reset_index(drop=True)
    return resultados, modelos_fit

In [21]:
df = df.sample(50_000, random_state=42)
(X_train, y_train, X_test, y_test), meta, d_train, d_test = build_features_reg(df)
print(meta)

{'feats': ['N', 'P', 'K', 'temperatura', 'umidade', 'pH', 'chuva', 'mes_sin', 'mes_cos', 'cultura_milho', 'cultura_soja', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste', 'regiao_Sul', 'estacao_outono', 'estacao_primavera', 'estacao_verao'], 'n_train': 40000, 'n_test': 10000, 'corte_timestamp': (Timestamp('2024-10-04 18:30:00'), Timestamp('2024-10-04 18:40:00')), 'cols_cat': ['cultura_milho', 'cultura_soja', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste', 'regiao_Sul', 'estacao_outono', 'estacao_primavera', 'estacao_verao']}


In [22]:
resultados, modelos_fit = treinar_modelos_reg(X_train, y_train, X_test, y_test, n_jobs=-1)
print(resultados)
resultados.to_csv("comparativo_modelos_reg.csv", index=False)

                      modelo        RMSE         MAE        R2    tempo_s
0      RandomForestRegressor  362.470484  244.562534  0.948354   3.614137
1  GradientBoostingRegressor  494.637382  346.141901  0.903824   5.874868
2        KNeighborsRegressor  571.013103  443.166877  0.871831   0.362744
3           LinearRegression  665.746133  498.655949  0.825775   0.047022
4                    SVR-RBF  970.330473  731.254320  0.629889  60.254092


In [23]:
def escolher_melhor_modelo(resultados, modelos_fit):
    # Ordena pelo menor RMSE, depois menor MAE, depois maior R2
    rank = resultados.sort_values(by=["RMSE", "MAE", "R2"], ascending=[True, True, False])
    melhor_nome = rank.iloc[0]["modelo"]
    melhor_modelo = modelos_fit[melhor_nome]
    return melhor_nome, melhor_modelo

In [24]:
melhor_nome, melhor_modelo = escolher_melhor_modelo(resultados, modelos_fit)
print("Melhor modelo:", melhor_nome)

Melhor modelo: RandomForestRegressor


# Extrair o perfil ideal

In [25]:
def extrair_perfil_ideal_reg(df_full: pd.DataFrame, mdl, feats: list, quantil=0.90):
    """
    Produz o perfil ideal de solo/clima para cada cultura
    usando o top 10% de produtividade prevista.
    Retorna um DataFrame com faixas (Q1–Q3) e medianas.
    """
    d = df_full.copy()

    # recriar mes_sin / mes_cos
    d["mes"] = pd.to_datetime(d["timestamp"]).dt.month
    d["mes_sin"] = np.sin(2 * np.pi * d["mes"] / 12)
    d["mes_cos"] = np.cos(2 * np.pi * d["mes"] / 12)

    # garantir que categorias existam
    d = pd.get_dummies(d, columns=["cultura", "regiao", "estacao"], drop_first=True)

    # garantir colunas na mesma ordem
    for col in feats:
        if col not in d.columns:
            d[col] = 0.0

    X = d[feats].to_numpy(dtype="float32")

    # previsões do melhor modelo
    d["prod_prevista"] = mdl.predict(X)

    perfis = []
    culturas = df_full["cultura"].unique()

    for c in culturas:
        mask = (df_full["cultura"] == c)
        sub = d.loc[mask].copy()

        thr = sub["prod_prevista"].quantile(quantil)
        elite = sub[sub["prod_prevista"] >= thr]

        def faixa(col):
            return f"{elite[col].quantile(0.25):.1f} — {elite[col].quantile(0.75):.1f} (med {elite[col].median():.1f})"

        perfis.append({
            "cultura": c,
            "N ideal": faixa("N"),
            "P ideal": faixa("P"),
            "K ideal": faixa("K"),
            "pH ideal": faixa("pH"),
            "temperatura ideal": faixa("temperatura"),
            "umidade ideal": faixa("umidade"),
            "chuva tolerada": f"≤ {elite['chuva'].quantile(0.75):.1f} mm",
            "prod mediana elite": elite["prod_prevista"].median().round(1),
        })

    return pd.DataFrame(perfis)

In [26]:
perfis_df = extrair_perfil_ideal_reg(df, melhor_modelo, meta["feats"])
perfis_df.to_csv("perfis_ideais_reg.csv", index=False)
perfis_df

,cultura,N ideal,P ideal,K ideal,pH ideal,temperatura ideal,umidade ideal,chuva tolerada,prod mediana elite
0,milho,61.7 — 74.9 (med 68.4),43.6 — 54.1 (med 49.3),54.8 — 65.1 (med 60.1),6.0 — 6.7 (med 6.4),22.7 — 27.2 (med 25.0),48.1 — 60.8 (med 54.0),≤ 2.7 mm,6383.2
1,cafe,61.3 — 74.9 (med 68.7),44.2 — 54.8 (med 49.8),54.8 — 64.5 (med 59.8),6.1 — 6.7 (med 6.4),22.6 — 27.0 (med 25.1),48.5 — 60.6 (med 54.1),≤ 2.6 mm,1523.5
2,soja,61.9 — 74.8 (med 68.1),44.3 — 55.0 (med 49.9),55.0 — 65.1 (med 60.1),6.1 — 6.7 (med 6.4),22.6 — 27.1 (med 24.9),47.7 — 60.7 (med 54.4),≤ 2.6 mm,3239.1
